# NB02 · Part 2 · Spark performance (pages 19–34)
Same rules as NB01: **Page N** cells are verbatim from the book, `# TEST SETUP` cells are not.

Attach the same test lakehouse, then *Run all*. On a small sample the timings mean little; this notebook checks that every card **runs**.

In [ ]:
%run NB00_Setup_Sample_Data

## Page 19 · How Spark runs your code

In [ ]:
# TEST SETUP (not in the book): inputs for the next card
df = sample_sales()

**Page 19 · Nothing runs until an action**

In [ ]:
# transformations: plan only, nothing runs
uk = df.filter(F.col("country") == "UK")
uk = uk.select("order_id", "amount")
# action: the job runs now
uk.count()

**Page 19 · See the plan before you tune**

In [ ]:
df.explain("formatted")
# "Exchange" in the plan = a shuffle

df.rdd.getNumPartitions()
spark.conf.get("spark.sql.shuffle.partitions")

## Page 20 · Spark UI

In [ ]:
# TEST SETUP (not in the book): inputs for the next card

def upsert_orders(src=None):
    spark.read.table("silver_orders").count()

def build_revenue():
    spark.read.table("silver_sales").groupBy("year").sum("amount").collect()

**Page 20 · Name your jobs so you can find them**

In [ ]:
sc = spark.sparkContext

sc.setJobDescription("silver: merge orders")
upsert_orders()

sc.setJobDescription("gold: revenue by month")
build_revenue()

## Page 22 · Set profiles

**Page 22 · For one notebook session**

In [ ]:
spark.conf.set("spark.fabric.resourceProfile",
               "readHeavyForPBI")

# what is active right now?
spark.conf.get("spark.fabric.resourceProfile")
spark.conf.get("spark.sql.parquet.vorder.default")

**Page 22 · V-Order on one table only**

In [ ]:
%%sql
ALTER TABLE gold_revenue SET TBLPROPERTIES
  ("delta.parquet.vorder.enabled" = "true")

In [ ]:
# TEST SETUP (not in the book): inputs for the next card

silver = sample_sales()
gold = silver.groupBy("country", "year").agg(F.sum("amount").alias("revenue"))

**Page 22 · One notebook writing two layers**

In [ ]:
PROFILE = "spark.fabric.resourceProfile"

spark.conf.set(PROFILE, "readHeavyForSpark")
(silver.write.mode("overwrite")
       .saveAsTable("silver_sales"))

spark.conf.set(PROFILE, "readHeavyForPBI")
(gold.write.mode("overwrite")
     .saveAsTable("gold_revenue"))

## Page 23 · V-Order and optimize write

**Page 23 · V-Order for one gold write**

In [ ]:
(gold.write.format("delta")
     .mode("overwrite")
     .option("parquet.vorder.enabled", "true")
     .saveAsTable("gold_revenue"))

**Page 23 · Is it on in this session?**

In [ ]:
%%sql
SET spark.sql.parquet.vorder.default

**Page 23 · Optimize write on one table**

In [ ]:
%%sql
ALTER TABLE silver_sales SET TBLPROPERTIES
  ('delta.autoOptimize.optimizeWrite' = 'true')

**Page 23 · Give old files V-Order**

In [ ]:
%%sql
OPTIMIZE gold_revenue VORDER

## Page 24 · Native Execution Engine

The `%%configure` card from page 24 is in NB03 (it restarts the session).

**Page 24 · Off for one query, then back on**

In [ ]:
%%sql
SET spark.native.enabled=FALSE;
-- run the query that misbehaves, then:
SET spark.native.enabled=TRUE;

In [ ]:
# TEST SETUP (not in the book): inputs for the next card
df = sample_sales().groupBy("country").sum("amount")

**Page 24 · Did it actually run natively?**

In [ ]:
df.explain()

## Page 25 · Small files and OPTIMIZE

**Page 25 · 1. Measure: average file size**

In [ ]:
d = spark.sql(
    "DESCRIBE DETAIL silver_sales").first()

files = max(d.numFiles, 1)
avg_mb = d.sizeInBytes / files / 1024**2
print(d.numFiles, "files,", round(avg_mb), "MB")

**Page 25 · 2. Know the target**

In [ ]:
key = ("spark.microsoft.delta"
       ".targetFileSize.adaptive.enabled")
spark.conf.set(key, "true")

**Page 25 · 3. Compact**

In [ ]:
%%sql
OPTIMIZE silver_sales;
OPTIMIZE silver_sales
  ZORDER BY (order_date, customer_id);
OPTIMIZE gold_revenue VORDER;

**Page 25 · 3. Compact**

In [ ]:
# the same from Python
from delta.tables import DeltaTable

DeltaTable.forName(spark, "silver_sales") \
    .optimize().executeCompaction()

**Page 25 · 4. Prevent: compact as you write**  
`%%sql` added so it runs as a SQL cell.

In [ ]:
%%sql
ALTER TABLE silver_sales SET TBLPROPERTIES
  ('delta.autoOptimize.autoCompact' = 'true')

## Page 26 · VACUUM

**Page 26 · VACUUM a table**

In [ ]:
%%sql
-- list what would go, delete nothing
VACUUM silver_sales RETAIN 168 HOURS DRY RUN;

VACUUM silver_sales RETAIN 168 HOURS;

**Page 26 · VACUUM a table**

In [ ]:
from delta.tables import DeltaTable
(DeltaTable.forName(spark, "silver_sales")
    .vacuum(168))

**Page 26 · A maintenance notebook**

In [ ]:
tables = [t.name
          for t in spark.catalog.listTables()
          if t.name.startswith(
              ("silver_", "gold_"))]

for name in tables:
    gold = name.startswith("gold_")
    vorder = " VORDER" if gold else ""
    spark.sql(f"OPTIMIZE {name}{vorder}")
    spark.sql(f"VACUUM {name} RETAIN 168 HOURS")
    print("done:", name)

## Page 27 · Data layout

In [ ]:
# TEST SETUP (not in the book): inputs for the next card
spark.sql("DROP TABLE IF EXISTS silver_orders")  # the card creates it

**Page 27 · Liquid clustering**

In [ ]:
%%sql
CREATE TABLE silver_orders
CLUSTER BY (order_date, region)
AS SELECT * FROM bronze_orders;

ALTER TABLE silver_orders
  CLUSTER BY (region, product);
OPTIMIZE silver_orders;   -- clusters here

**Page 27 · Z-order only recent partitions**

In [ ]:
%%sql
-- WHERE works on partition columns only
OPTIMIZE gold_sales_history
WHERE year = 2026
ZORDER BY (order_date, region)

In [ ]:
# TEST SETUP (not in the book): inputs for the next card

(spark.read.table("orders").filter("region = 'NA'").limit(20)
      .withColumn("amount", F.col("amount") * 2)
 .createOrReplaceTempView("staged"))

**Page 27 · Partitioned MERGE: name the slice**  
`%%sql` added so it runs as a SQL cell.

In [ ]:
%%sql
MERGE INTO orders t USING staged s
ON t.order_id = s.order_id
   AND t.region = 'NA' AND s.region = 'NA'
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

## Page 28 · Shuffles and AQE

**Page 28 · AQE: let Spark fix it at run time**

In [ ]:
for k in ["enabled",
          "coalescePartitions.enabled",
          "skewJoin.enabled"]:
    key = "spark.sql.adaptive." + k
    print(key, spark.conf.get(key))

**Page 28 · Set shuffle partitions by hand**

In [ ]:
# default 200; AQE coalesces down, never up
spark.conf.set("spark.sql.shuffle.partitions",
               400)

**Page 28 · Shuffle less data**

In [ ]:
result = (orders
  .filter(F.col("year") == 2026)    # 1. filter
  .select("customer_id", "amount")  # 2. columns
  .groupBy("customer_id")           # 3. shuffle
  .agg(F.sum("amount").alias("total")))

In [ ]:
# TEST SETUP (not in the book): inputs for the next card
spark.conf.set("spark.sql.shuffle.partitions", 200)  # back to the default

## Page 29 · Broadcast joins

**Page 29 · Ask for a broadcast**

In [ ]:
df = orders.join(F.broadcast(dim_country),
                 "country_code", "left")

**Page 29 · Ask for a broadcast**  
`%%sql` added so it runs as a SQL cell.

In [ ]:
%%sql
SELECT /*+ BROADCAST(c) */ o.*, c.country_name
FROM   orders o
JOIN   dim_country c
  ON   o.country_code = c.country_code

**Page 29 · The automatic threshold**

In [ ]:
key = "spark.sql.autoBroadcastJoinThreshold"
spark.conf.get(key)              # 10 MB default
spark.conf.set(key, 50*1024**2)  # raise: 50 MB
spark.conf.set(key, -1)          # auto off

In [ ]:
# TEST SETUP (not in the book): inputs for the next card
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")  # back to the default

**Page 29 · Make the small side smaller**

In [ ]:
dim = (spark.read.table("dim_customer")
         .filter("is_current = true")
         .select("customer_id", "segment"))

df = orders.join(F.broadcast(dim),
                 "customer_id")

## Page 30 · Data skew

**Page 30 · Find the heavy keys**

In [ ]:
(orders.groupBy("customer_id")
       .count()
       .orderBy(F.desc("count"))
       .show(10))

**Page 30 · Four fixes, cheapest first**

In [ ]:
spark.conf.get(
    "spark.sql.adaptive.skewJoin.enabled")
# skewed = 5x median AND over 256 MB, by default

**Page 30 · Four fixes, cheapest first**

In [ ]:
key = F.col("customer_id")
nulls = orders.filter(key.isNull())
rest = (orders.filter(key.isNotNull())
        .join(customers, "customer_id", "left"))

result = rest.unionByName(
    nulls, allowMissingColumns=True)

**Page 30 · Four fixes, cheapest first**

In [ ]:
N = 8   # how many ways to split

# big side: random salt 0..N-1 per row
big = orders.withColumn(
    "salt", (F.rand() * N).cast("int"))

# small side: one copy per salt value
salts = spark.range(N).select(
    F.col("id").cast("int").alias("salt"))
small = customers.crossJoin(salts)

result = (big.join(small, ["customer_id", "salt"])
             .drop("salt"))

## Page 31 · Caching

In [ ]:
# TEST SETUP (not in the book): inputs for the next card

def build_enriched():
    return orders.join(F.broadcast(dim_country), "country_code", "left")

def write_silver(d): d.limit(10).count()
def build_gold(d): d.groupBy("country_name").sum("amount").collect()
def run_checks(d): assert d.filter("order_id IS NULL").isEmpty()

**Page 31 · Cache, use, release**

In [ ]:
enriched = build_enriched().cache()
enriched.count()            # fill the cache

write_silver(enriched)
build_gold(enriched)
run_checks(enriched)

enriched.unpersist()        # free the memory

**Page 31 · Very long plans: save a checkpoint table**

In [ ]:
enriched.write.mode("overwrite") \
    .saveAsTable("stg_enriched")
enriched = spark.read.table("stg_enriched")

## Page 32 · Anti-patterns
Each pair is *slow* (orange in the book) then *better* (green). Both should run.

In [ ]:
# TEST SETUP (not in the book): inputs for the next card
df = spark.read.table("customers")

**Page 32 · 1. A Python UDF for something Spark already does**

In [ ]:
clean = F.udf(lambda s: s.strip().upper())
df = df.withColumn("name", clean("name"))

**Page 32 · 1. A Python UDF for something Spark already does**

In [ ]:
df = df.withColumn("name",
       F.upper(F.trim("name")))

In [ ]:
# TEST SETUP (not in the book): inputs for the next card
df = spark.read.table("customers")
text_cols = ["name", "region"]

**Page 32 · 2. withColumn in a loop**

In [ ]:
for c in text_cols:
    df = df.withColumn(c, F.trim(c))

**Page 32 · 2. withColumn in a loop**

In [ ]:
df = df.select([F.trim(c).alias(c)
                if c in text_cols else c
                for c in df.columns])

**Page 32 · 3. Pulling a big table onto the driver**

In [ ]:
pdf = spark.read.table("silver_sales").toPandas()
top = pdf.groupby("country")["amount"].sum()

**Page 32 · 3. Pulling a big table onto the driver**

In [ ]:
top = (spark.read.table("silver_sales")
         .groupBy("country").sum("amount")
         .toPandas())   # small result only

In [ ]:
# TEST SETUP (not in the book): inputs for the next card
updates = spark.createDataFrame([(1, "gold"), (2, "silver")], "id int, tier string")

**Page 32 · 4. Row-by-row loops instead of a join**

In [ ]:
for row in updates.collect():
    spark.sql(f"UPDATE customers SET tier = "
              f"'{row.tier}' WHERE id = {row.id}")

**Page 32 · 4. Row-by-row loops instead of a join**

In [ ]:
(DeltaTable.forName(spark, "customers").alias("t")
   .merge(updates.alias("s"), "t.id = s.id")
   .whenMatchedUpdate(set={"tier": "s.tier"})
   .execute())

In [ ]:
# TEST SETUP (not in the book): inputs for the next card

new_rows = spark.read.table("orders").filter("order_date = '2026-10-19'")

def load(d): print("loaded", d.count(), "rows")

**Page 32 · 5. Counting everything to check for anything**

In [ ]:
if new_rows.count() > 0:
    load(new_rows)

**Page 32 · 5. Counting everything to check for anything**

In [ ]:
if not new_rows.isEmpty():   # stops at 1 row
    load(new_rows)

In [ ]:
# TEST SETUP (not in the book): inputs for the next card

old = spark.read.table("silver_sales").filter("order_date <> '2026-10-19'").drop("channel")
today = sample_sales().filter("order_date = '2026-10-19'")

**Page 32 · 6. Rewriting the whole table for one day's change**

In [ ]:
full = old.union(today)
full.write.mode("overwrite") \
    .saveAsTable("silver_sales")

**Page 32 · 6. Rewriting the whole table for one day's change**

In [ ]:
(today.write.mode("overwrite")
   .option("replaceWhere",
           "order_date = '2026-10-19'")
   .saveAsTable("silver_sales"))

## Page 33 · Pools
The `session.stop()` card is in NB03. The high-concurrency setting is an Environment property, not notebook code: check it in *Environment › Spark properties*.